# NB3 — Master Classifier Evaluation, Stability & Significance

> **Release note:** this notebook is **NB3** in the paper (development name: Notebook 4). Its output files keep the prefix `nb4_` (for example `nb4_stability.csv`) so that they match the released result files. Some saved outputs were cleared for this release; rerunning the notebook regenerates them.
### Healthcare-Security Feature-Selection Benchmark · Four-Dataset Scientific Master

**Version 4.2 — stability/checkpoint/RUNS integration hardened**

This version preserves the scientific protocol of NB3 while fixing the bookkeeping and checkpoint issues identified in v4.1.

**Inputs**
- Notebook 2: canonical **30 baseline feature-selection methods** and `nb2_selection_matrix.csv`.

**Datasets**: HIPAA, WUSTL, CICIOMT, WUSTL_HDRL.

**Primary evaluation**
- Held-out test performance: MCC, AUC, balanced accuracy, macro/weighted F1, precision, recall, accuracy, Cohen's kappa.
- Training-only CV for classifier selection: MCC is primary.
- SMOTE is applied inside CV folds only and to training data only before final held-out testing.
- Selection stability: Jaccard and Nogueira stability using the original method-specific selectors; no proxy selector aliases are used.
- Compactness: selected-feature count.
- Per-dataset Friedman/Nemenyi is exploratory; cross-dataset inference is reserved for NB4.

**v4.2 engineering safeguards**
1. `RUNS` has an explicit 12-item schema: indices 0–9 evaluation package, 10 = CV package, 11 = stability DataFrame.
2. Stability insertion preserves the existing CV package instead of overwriting it.
3. Stability checkpoints resume from the existing v4.1 checkpoint when its deterministic bootstrap design is compatible, while writing the v4.2 protocol metadata.
4. Partial bootstrap results are resumable per method/per bootstrap.
5. Failed bootstrap attempts are retried; stale failure records are removed after success.
6. Stability audit requires all **30** methods to have `STABILITY_K` successful bootstraps before PASS.
7. Audit distinguishes `PASS`, `PARTIAL`, and `FAIL` rather than labeling incomplete stability as a generic failure.
8. Evaluation/CV checkpoint protocol identifiers remain compatible with the existing v3 checkpoints so completed work is not unnecessarily recomputed.


## Section 0 — Imports and protocol

In [ ]:
import os, json, time, traceback, warnings, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
    precision_score, recall_score, matthews_corrcoef, cohen_kappa_score,
    roc_auc_score, make_scorer, confusion_matrix, roc_curve)
from sklearn.pipeline import Pipeline

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    IMB_AVAILABLE = True
except Exception:
    IMB_AVAILABLE = False
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
warnings.filterwarnings('ignore')
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 40)

# -----------------------------------------------------------------------------
# Protocol identifiers
# -----------------------------------------------------------------------------
PROTOCOL_VERSION = 'NB4_MASTER_SCIENTIFIC_v4_2_NB2_NB3'
# Existing v3 evaluation/CV checkpoints remain valid and are intentionally reused.
EVALUATION_PROTOCOL_VERSION = 'NB4_MASTER_SCIENTIFIC_v3_NB2_NB3'
CV_PROTOCOL_VERSION = 'NB4_MASTER_SCIENTIFIC_v3_NB2_NB3'
STABILITY_PROTOCOL_VERSION = 'NB4_MASTER_SCIENTIFIC_v4_2_EXACT_STABILITY'
LEGACY_STABILITY_PROTOCOL_VERSION = 'NB4_MASTER_SCIENTIFIC_v4_1_EXACT_STABILITY'
RUNS_SCHEMA_VERSION = 'NB4_RUNS_SCHEMA_v4_2'

DATASETS = ['HIPAA','WUSTL','CICIOMT','WUSTL_HDRL']
RANDOM_STATE = 42
CV_FOLDS = 5
PRIMARY_METRIC = 'mcc'
IMBALANCE = 'smote'
N_JOBS = 1

# -----------------------------------------------------------------------------
# Resource-safe controls
# -----------------------------------------------------------------------------
EVAL_MAX_ROWS = None
STABILITY_MAX_ROWS = 5000
STABILITY_K = 30
RUN_STABILITY = True
STAB_INCLUDE_HEAVY = True
# None => min(20, n_features); otherwise use the requested population size.
STABILITY_META_POP = None
STABILITY_META_ITERS = 10
BORUTASHAP_STAB_TRIALS = 10
STAB_N_JOBS = 1
K_STAB = 12

# Checkpointing
RESUME = True
SAVE_CHECKPOINT_EVERY_RESULT = True
STABILITY_CHECKPOINT_NAME = 'nb4_stability_v4_checkpoint.json'

CANONICAL_BASELINE_30 = [
'chi2','anova','mutual_info','info_gain','gain_ratio','fisher','reliefF','variance','spearman','fcbf','laplacian',
'rfe','rfecv','sfs','sbs','ga','bpso','bgwo',
'lasso','elasticnet','rf_importance','extratrees_importance','dtree_importance','xgb_importance','lgbm_importance','permutation',
'mrmr','boruta','borutashap','shap_rank'
]
EXPECTED_METHOD_COUNT = 30
EXPECTED_STABILITY_METHODS = list(CANONICAL_BASELINE_30)
EXPECTED_STABILITY_METHOD_COUNT = 30
EXPECTED_CLASSIFIER_COUNT = 4
EXPECTED_RUNS_LEN = 12

BENCH_ROOT = r'C:\Users\tanvi\Downloads\HealthSec_FS'
print('Protocol:', PROTOCOL_VERSION)
print('Evaluation checkpoint protocol:', EVALUATION_PROTOCOL_VERSION)
print('CV checkpoint protocol:', CV_PROTOCOL_VERSION)
print('Stability protocol:', STABILITY_PROTOCOL_VERSION)
print('Datasets:', DATASETS)
print('Canonical baseline methods:', len(CANONICAL_BASELINE_30))
print('Expected stability methods:', EXPECTED_STABILITY_METHOD_COUNT)
print('IMB available:', IMB_AVAILABLE)
print('XGB available:', XGB_AVAILABLE)


## Section 1 — Utility functions and safe target handling

In [2]:
def safe_target(y_train, y_test):
    yt = y_train.copy(); ye = y_test.copy()
    if yt.dtype == object or str(yt.dtype).startswith('category'):
        le = LabelEncoder().fit(yt.astype(str))
        yt = pd.Series(le.transform(yt.astype(str)), index=yt.index)
        # Reject unseen test labels rather than silently mis-encoding them.
        vals = set(ye.astype(str)) - set(le.classes_)
        if vals:
            raise ValueError(f'Test contains unseen target labels: {sorted(vals)[:5]}')
        ye = pd.Series(le.transform(ye.astype(str)), index=ye.index)
    else:
        # Normalize boolean / numeric binary targets.
        if yt.dtype == bool:
            yt = yt.astype(int); ye = ye.astype(int)
        elif not pd.api.types.is_numeric_dtype(yt):
            le = LabelEncoder().fit(yt.astype(str)); yt = pd.Series(le.transform(yt.astype(str)), index=yt.index); ye = pd.Series(le.transform(ye.astype(str)), index=ye.index)
    classes = sorted(pd.Series(yt).dropna().unique().tolist())
    if len(classes) != 2:
        raise ValueError(f'Only binary classification is supported; found classes={classes}')
    return yt.astype(int), ye.astype(int), classes

def clean_features(Xtr, Xte):
    Xtr = Xtr.copy(); Xte = Xte.copy()
    # Convert booleans to numeric.
    for c in Xtr.columns:
        if Xtr[c].dtype == bool:
            Xtr[c] = Xtr[c].astype(int); Xte[c] = Xte[c].astype(int)
    # One-hot encode remaining non-numeric columns jointly, then split.
    nonnum = [c for c in Xtr.columns if not pd.api.types.is_numeric_dtype(Xtr[c])]
    if nonnum:
        both = pd.concat([Xtr, Xte], axis=0)
        both = pd.get_dummies(both, columns=nonnum, dummy_na=True)
        Xtr = both.iloc[:len(Xtr)].copy(); Xte = both.iloc[len(Xtr):].copy()
    Xtr = Xtr.replace([np.inf,-np.inf], np.nan); Xte = Xte.replace([np.inf,-np.inf], np.nan)
    # Fit medians on training only.
    med = Xtr.median(numeric_only=True)
    Xtr = Xtr.fillna(med).fillna(0); Xte = Xte.fillna(med).fillna(0)
    # Remove constants using training only.
    keep = [c for c in Xtr.columns if Xtr[c].nunique(dropna=False) > 1]
    if not keep:
        raise ValueError('No non-constant numeric features remain after cleaning.')
    return Xtr[keep], Xte.reindex(columns=keep, fill_value=0)

def subset_hash(subsets):
    payload = json.dumps({k: sorted(v) for k,v in subsets.items()}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

def stable_hash(values):
    payload = json.dumps([str(v) for v in values], ensure_ascii=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

def write_json(path, payload):
    # Atomic replace prevents a kernel interruption from leaving a half-written checkpoint.
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf8') as f:
        json.dump(payload, f, indent=2)
    os.replace(tmp, path)

def smote_k(y):
    n = int(pd.Series(y).value_counts().min())
    if n < 2: raise ValueError('SMOTE requires at least two samples in every class.')
    return max(1, min(5, n-1))

def metric_dict(yt, yp, score):
    out = {
        'accuracy': accuracy_score(yt,yp),
        'f1_weighted': f1_score(yt,yp,average='weighted',zero_division=0),
        'f1_macro': f1_score(yt,yp,average='macro',zero_division=0),
        'balanced_acc': balanced_accuracy_score(yt,yp),
        'precision_macro': precision_score(yt,yp,average='macro',zero_division=0),
        'recall_macro': recall_score(yt,yp,average='macro',zero_division=0),
        'mcc': matthews_corrcoef(yt,yp),
        'kappa': cohen_kappa_score(yt,yp),
    }
    try: out['auc'] = roc_auc_score(yt, score)
    except Exception: out['auc'] = np.nan
    return out

def score_vector(clf, X):
    if hasattr(clf,'predict_proba'): return clf.predict_proba(X)[:,1]
    if hasattr(clf,'decision_function'): return clf.decision_function(X)
    return clf.predict(X)


## Section 2 — Dataset loader and NB2 integration

In [3]:
def load_xy(ds):
    root = os.path.join(BENCH_ROOT, ds)
    train_path = os.path.join(root,'processed','train.parquet')
    test_path  = os.path.join(root,'processed','test.parquet')
    if not os.path.exists(train_path): raise FileNotFoundError(train_path)
    if not os.path.exists(test_path): raise FileNotFoundError(test_path)
    tr = pd.read_parquet(train_path); te = pd.read_parquet(test_path)
    target = 'is_attack'
    if target not in tr.columns or target not in te.columns: raise KeyError(f'{target} missing in {ds}')
    ytr,yte,classes = safe_target(tr[target],te[target])
    Xtr = tr.drop(columns=[target]); Xte = te.drop(columns=[target])
    Xtr,Xte = clean_features(Xtr,Xte)
    return Xtr,Xte,ytr,yte,classes,train_path,test_path

def load_subsets(ds, feature_space):
    results = os.path.join(BENCH_ROOT, ds, 'results')
    subsets = {'ALL_FEATURES': list(feature_space)}
    provenance = {'ALL_FEATURES':'reference'}
    nb2_path = os.path.join(results,'nb2_selection_matrix.csv')
    if not os.path.exists(nb2_path): raise FileNotFoundError(nb2_path)
    mat = pd.read_csv(nb2_path,index_col=0)
    missing = [m for m in CANONICAL_BASELINE_30 if m not in mat.index]
    if missing: raise AssertionError(f'{ds}: NB2 missing canonical methods: {missing}')
    for m in CANONICAL_BASELINE_30:
        feats = [c for c in mat.columns if c in feature_space and pd.to_numeric(pd.Series([mat.loc[m,c]]),errors='coerce').iloc[0] == 1]
        if not feats: raise AssertionError(f'{ds}: empty NB2 subset for {m}')
        subsets[m]=feats; provenance[m]='NB2 baseline'
    return subsets,provenance,{}


## Section 3 — Classifiers and leakage-safe evaluation

In [4]:
def classifier_factories(ytr):
    # Resource-safe single-threaded models for Windows/local Jupyter.
    return {
      'SVM': lambda: SVC(kernel='rbf', probability=False, class_weight='balanced' if IMBALANCE=='class_weight' else None, random_state=RANDOM_STATE),
      'RandomForest': lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=N_JOBS, class_weight='balanced' if IMBALANCE=='class_weight' else None),
      'LogisticRegression': lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced' if IMBALANCE=='class_weight' else None),
      'XGBoost': lambda: __import__('xgboost').XGBClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=N_JOBS, tree_method='hist', eval_metric='logloss')
    }

def make_estimator(factory,y):
    clf=factory()
    if IMBALANCE=='smote':
        if not IMB_AVAILABLE: raise ImportError('imbalanced-learn is required for IMBALANCE=smote')
        return ImbPipeline([('smote',SMOTE(random_state=RANDOM_STATE,k_neighbors=smote_k(y))),('clf',clf)])
    return clf

def cap_training(X,y):
    if EVAL_MAX_ROWS is None or len(X)<=EVAL_MAX_ROWS: return X,y
    idx,_=train_test_split(np.arange(len(X)),train_size=EVAL_MAX_ROWS,stratify=y,random_state=RANDOM_STATE)
    return X.iloc[idx].copy(), y.iloc[idx].copy()

def fit_test(estimator,Xtr,ytr,Xte,yte):
    estimator.fit(Xtr,ytr)
    yp=estimator.predict(Xte); sc=score_vector(estimator,Xte)
    return metric_dict(yte,yp,sc),yp,sc


## Section 4 — Four-dataset evaluation with checkpoints

In [ ]:
def run_evaluation(ds):
    Xtr,Xte,ytr,yte,classes,train_path,test_path=load_xy(ds)
    subsets,provenance,tri=load_subsets(ds,list(Xtr.columns))
    Xev,yev=cap_training(Xtr,ytr)
    factories=classifier_factories(ytr)
    outdir=os.path.join(BENCH_ROOT,ds,'results'); os.makedirs(outdir,exist_ok=True)
    ckpt_path=os.path.join(outdir,'nb4_evaluation_checkpoint.json')
    state=json.load(open(ckpt_path,encoding='utf8')) if RESUME and os.path.exists(ckpt_path) else {'protocol_version':EVALUATION_PROTOCOL_VERSION,'completed':{},'failures':{}}
    if state.get('protocol_version')!=EVALUATION_PROTOCOL_VERSION:
        state={'protocol_version':EVALUATION_PROTOCOL_VERSION,'completed':{},'failures':{}}
    records=[]; store={}
    if state.get('completed'):
        records=[r for r in state['completed'].values() if r.get('subset') in subsets]
    done=set(state.get('completed',{}).keys())
    t0=time.time()
    for sname,feats in subsets.items():
        for cname,factory in factories.items():
            key=f'{sname}||{cname}'
            if key in done: continue
            try:
                est=make_estimator(factory,yev)
                mets,yp,sc=fit_test(est,Xev[feats],yev,Xte[feats],yte)
                rec={'dataset':ds,'subset':sname,'classifier':cname,'n_features':len(feats),'provenance':provenance.get(sname,'unknown'),'eval_rows':len(Xev)}; rec.update(mets)
                records.append(rec); state['completed'][key]=rec
                state.setdefault('failures', {}).pop(key, None)
                store[key]=(yp,sc)
                print(f'  [{sname:<28} / {cname:<18}] MCC={mets["mcc"]:.4f} AUC={mets["auc"]:.4f}')
            except Exception as e:
                state['failures'][key]={'error_type':type(e).__name__,'error':str(e),'traceback':traceback.format_exc(limit=6)}
                print(f'  ERROR {sname}/{cname}: {type(e).__name__}: {e}')
            if SAVE_CHECKPOINT_EVERY_RESULT:
                write_json(ckpt_path, state)
    master=pd.DataFrame(records)
    master.to_csv(os.path.join(outdir,'nb4_master_comparison.csv'),index=False)
    return Xtr,Xte,ytr,yte,subsets,provenance,tri,master,state,store

RUNS={}; RUN_AUDITS=[]
for ds in DATASETS:
    print('\n'+'='*78); print('NB4 ::',ds); print('='*78)
    try:
        RUNS[ds]=run_evaluation(ds)
        master=RUNS[ds][7]; state=RUNS[ds][8]
        print(f'[{ds}] evaluation rows={len(master)} failures={len(state.get("failures",{}))}')
    except Exception as e:
        RUN_AUDITS.append({'dataset':ds,'status':'FAIL','error_type':type(e).__name__,'error':str(e)})
        print(f'[{ds}] DATASET FAILED: {type(e).__name__}: {e}')
        print(traceback.format_exc(limit=6))


## Section 5 — Training-only CV classifier selection and held-out test reporting

In [ ]:
def cv_select(ds):
    if ds not in RUNS: return None
    Xtr,Xte,ytr,yte,subsets,provenance,tri,master,state,store=RUNS[ds]
    factories=classifier_factories(ytr); CV=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=RANDOM_STATE)
    rows=[]
    outdir=os.path.join(BENCH_ROOT,ds,'results'); path=os.path.join(outdir,'nb4_classifier_selection_cv.csv')
    ck=os.path.join(outdir,'nb4_cv_checkpoint.json')
    cstate=json.load(open(ck,encoding='utf8')) if RESUME and os.path.exists(ck) else {'protocol_version':CV_PROTOCOL_VERSION,'completed':{},'failures':{}}
    if cstate.get('protocol_version')!=CV_PROTOCOL_VERSION: cstate={'protocol_version':CV_PROTOCOL_VERSION,'completed':{},'failures':{}}
    scoring={'mcc':make_scorer(matthews_corrcoef),'balanced_acc':make_scorer(balanced_accuracy_score),'f1_macro':make_scorer(f1_score,average='macro',zero_division=0),'auc':'roc_auc'}
    for sname,feats in subsets.items():
        for cname,factory in factories.items():
            key=f'{sname}||{cname}'
            if key in cstate['completed']: rows.append(cstate['completed'][key]); continue
            try:
                est=make_estimator(factory,ytr)
                cvres=cross_validate(est,Xtr[feats],ytr,cv=CV,scoring=scoring,n_jobs=N_JOBS,error_score=np.nan,return_train_score=False)
                if not np.isfinite(cvres['test_mcc']).any():
                    raise RuntimeError('CV produced no finite MCC values; combination is not usable.')
                rec={'dataset':ds,'subset':sname,'classifier':cname,'n_features':len(feats),
                     'cv_mean_mcc':float(np.nanmean(cvres['test_mcc'])),'cv_std_mcc':float(np.nanstd(cvres['test_mcc'],ddof=1)),
                     'cv_mean_auc':float(np.nanmean(cvres['test_auc'])),'cv_mean_f1_macro':float(np.nanmean(cvres['test_f1_macro'])),'cv_mean_balanced_acc':float(np.nanmean(cvres['test_balanced_acc']))}
                # Store individual folds for later reproducibility.
                for i,v in enumerate(cvres['test_mcc']): rec[f'mcc_fold_{i+1}']=float(v)
                for i,v in enumerate(cvres['test_auc']): rec[f'auc_fold_{i+1}']=float(v)
                cstate['completed'][key]=rec; cstate.setdefault('failures', {}).pop(key, None); rows.append(rec)
            except Exception as e:
                cstate.setdefault('failures',{})[key]={'error_type':type(e).__name__,'error':str(e)}
            write_json(ck, cstate)
    cvdf=pd.DataFrame(rows); cvdf.to_csv(path,index=False)
    # One classifier per subset, selected using training-only CV MCC.
    valid_cv=cvdf[np.isfinite(pd.to_numeric(cvdf.get('cv_mean_mcc', pd.Series(dtype=float)), errors='coerce'))].copy()
    bestclf=(valid_cv.sort_values(['subset','cv_mean_mcc'],ascending=[True,False]).groupby('subset',as_index=False).first() if not valid_cv.empty else pd.DataFrame(columns=['subset','classifier','cv_mean_mcc','cv_std_mcc','cv_mean_auc','cv_mean_f1_macro','cv_mean_balanced_acc']))
    cols=['dataset','subset','classifier','n_features','provenance','accuracy','f1_weighted','f1_macro','balanced_acc','precision_macro','recall_macro','mcc','kappa','auc','eval_rows']
    testdf=master[[c for c in cols if c in master.columns]].copy()
    best=testdf.merge(bestclf[['subset','classifier','cv_mean_mcc','cv_std_mcc','cv_mean_auc','cv_mean_f1_macro','cv_mean_balanced_acc']],on=['subset','classifier'],how='inner')
    best=best.sort_values('cv_mean_mcc',ascending=False)
    best.to_csv(os.path.join(outdir,'nb4_best_per_subset.csv'),index=False)
    # Save selected-classifier fold scores for Notebook 5.
    fold_rows=[]
    for _,r in bestclf.iterrows():
        rr=r.to_dict(); fold_rows.append(rr)
    pd.DataFrame(fold_rows).to_csv(os.path.join(outdir,'nb4_perfold_cv.csv'),index=False)
    print(f'[{ds}] CV complete: {len(cvdf)} subset-classifier combinations; selected {len(best)} subsets.')
    return cvdf,best,bestclf

for ds in list(RUNS):
    try:
        RUNS[ds]=RUNS[ds]+(cv_select(ds),)
    except Exception as e:
        print(f'[{ds}] CV selection ERROR: {type(e).__name__}: {e}')


## Section 6 — Exploratory per-dataset significance

In [ ]:
from scipy.stats import friedmanchisquare, studentized_range

def nemenyi_cd(k,n,alpha=0.05):
    q=studentized_range.ppf(1-alpha,k,np.inf)/np.sqrt(2)
    return float(q*np.sqrt(k*(k+1)/(6*n)))

def per_dataset_stats(ds):
    if ds not in RUNS or len(RUNS[ds])<11: return pd.DataFrame()
    cvpack=RUNS[ds][10]
    best=cvpack[1]
    panel=best[best['subset'].isin(CANONICAL_BASELINE_30)].copy()
    if panel.empty: return pd.DataFrame()
    # One observation per method is insufficient for Friedman; use the 5 CV folds from the selected classifier.
    cvdf=cvpack[0]
    sel=cvpack[2]
    selected=sel[sel['subset'].isin(CANONICAL_BASELINE_30)]
    foldcols=[f'mcc_fold_{i}' for i in range(1,CV_FOLDS+1)]
    available=[c for c in foldcols if c in selected.columns]
    if len(available)<2 or selected['subset'].nunique()<2: return pd.DataFrame()
    mat=selected.set_index('subset')[available].T
    mat=mat.dropna(axis=0,how='any')
    if mat.shape[0]<2 or mat.shape[1]<2: return pd.DataFrame()
    stat,p=friedmanchisquare(*[mat[c].values for c in mat.columns])
    cd=nemenyi_cd(mat.shape[1],mat.shape[0])
    rec={'dataset':ds,'metric':'CV_MCC_EXPLORATORY_FOLDS','n_methods':mat.shape[1],'n_folds':mat.shape[0],'friedman_chi2':float(stat),'p_value':float(p),'nemenyi_cd':cd}
    out=pd.DataFrame([rec]); out.to_csv(os.path.join(BENCH_ROOT,ds,'results','nb4_friedman_nemenyi.csv'),index=False); return out

STATS=[]
for ds in RUNS:
    try:
        z=per_dataset_stats(ds)
        if not z.empty: STATS.append(z.iloc[0].to_dict()); print(z.to_string(index=False))
    except Exception as e: print(f'[{ds}] significance ERROR: {type(e).__name__}: {e}')


## Section 7 — Selection stability: Jaccard + Nogueira, checkpointed — exact method-specific selectors


In [8]:
print("========== RUNS SCHEMA GATE: BEFORE STABILITY ==========")
for ds in DATASETS:
    if ds not in RUNS:
        print(f"{ds}: NOT IN RUNS")
        continue
    n = len(RUNS[ds])
    print(f"{ds}: tuple length={n} (expected 11 before stability)")
    assert n == 11, f"{ds}: unexpected RUNS length before stability: {n}"
    assert isinstance(RUNS[ds][10], tuple), f"{ds}: CV package missing at RUNS[{ds}][10]"
print("RUNS pre-stability schema gate: PASS")


========== RUNS SCHEMA GATE: BEFORE STABILITY ==========
HIPAA: tuple length=11 (expected 11 before stability)
WUSTL: tuple length=11 (expected 11 before stability)
CICIOMT: tuple length=11 (expected 11 before stability)
WUSTL_HDRL: tuple length=11 (expected 11 before stability)
RUNS pre-stability schema gate: PASS


In [ ]:
import os
import sys

# Prevent OpenMP / Joblib thread deadlocks in Jupyter before importing numerical backends
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import subprocess
import traceback
import warnings
from scipy.stats import spearmanr

import numpy as np
import pandas as pd

packages = ["scikit-rebate", "Boruta", "lightgbm", "mrmr-selection", "zoofs", "BorutaShap"]
from sklearn.utils import resample

try:
    import boruta
    import lightgbm
    import mrmr
    import skrebate

    print("All required stability-selector packages are already installed.")
except ImportError:
    print("Installing missing stability-selector dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
    print("Installation complete.")

from mlxtend.feature_selection import SequentialFeatureSelector as _SFS
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import (
    RFE,
    RFECV,
    SelectFromModel,
    chi2,
    f_classif,
    mutual_info_classif,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

# Execution controls are inherited from Section 0 so there is one source of truth.
STAB_N_JOBS = globals().get("STAB_N_JOBS", 1)
K_STAB = globals().get("K_STAB", 12)
STAB_INCLUDE_HEAVY = globals().get("STAB_INCLUDE_HEAVY", True)
STABILITY_META_POP = globals().get("STABILITY_META_POP", None)
STABILITY_META_ITERS = globals().get("STABILITY_META_ITERS", 10)
BORUTASHAP_STAB_TRIALS = globals().get("BORUTASHAP_STAB_TRIALS", 10)


# ---- Stability Metric Helper Functions ----
def jaccard_index(a, b):
    a, b = set(a), set(b)
    u = a | b
    return len(a & b) / len(u) if u else 1.0


def mean_pairwise_jaccard(subs):
    if len(subs) < 2:
        return np.nan
    vals = [
        jaccard_index(subs[i], subs[j])
        for i in range(len(subs))
        for j in range(i + 1, len(subs))
    ]
    return float(np.mean(vals)) if vals else np.nan


def nogueira_stability(subs, feature_space):
    if len(subs) < 2:
        return np.nan
    idx = {f: i for i, f in enumerate(feature_space)}
    Z = np.zeros((len(subs), len(feature_space)), float)
    for r, s in enumerate(subs):
        for f in s:
            if f in idx:
                Z[r, idx[f]] = 1
    M, d = Z.shape
    if d == 0:
        return np.nan
    p = Z.mean(0)
    kbar = p.sum()
    den = (kbar / d) * (1 - kbar / d)
    if den <= 0:
        return 1.0
    num = (M / (M - 1)) * np.sum(p * (1 - p)) / d
    return float(1 - num / den)


def _topk(Xs, sc, k=K_STAB):
    sc = np.nan_to_num(np.asarray(sc, dtype=float))
    return [Xs.columns[i] for i in np.argsort(sc)[::-1][: min(k, Xs.shape[1])]]


def _ent(y):
    _, c = np.unique(y, return_counts=True)
    p = c / c.sum()
    return float(-(p * np.log2(p + 1e-12)).sum())


def _disc(x, bins=8):
    u = np.unique(x)
    if len(u) <= bins:
        return np.searchsorted(u, x)
    return np.digitize(x, np.histogram_bin_edges(x, bins=bins)[1:-1])


# ---- Filter Methods (9) ----
def s_chi2(Xs, ys, k=K_STAB):
    sc, _ = chi2(MinMaxScaler().fit_transform(Xs), ys)
    return _topk(Xs, sc, k)


def s_anova(Xs, ys, k=K_STAB):
    F, _ = f_classif(Xs, ys)
    return _topk(Xs, F, k)


def s_mi(Xs, ys, k=K_STAB):
    return _topk(Xs, mutual_info_classif(Xs, ys, random_state=RANDOM_STATE), k)


def s_variance(Xs, ys, k=K_STAB):
    return _topk(Xs, Xs.var().values, k)


def s_spearman(Xs, ys, k=K_STAB):
    yv = ys.values
    sc = [
        abs(np.nan_to_num(spearmanr(Xs[c].values, yv).correlation))
        for c in Xs.columns
    ]
    return _topk(Xs, sc, k)


def s_fisher(Xs, ys, k=K_STAB):
    yv = ys.values
    cls = np.unique(yv)
    out = []
    for c in Xs.columns:
        xc = Xs[c].values
        mu = xc.mean()
        num = den = 0.0
        for cl in cls:
            xi = xc[yv == cl]
            num += len(xi) * (xi.mean() - mu) ** 2
            den += len(xi) * xi.var()
        out.append(num / (den + 1e-12))
    return _topk(Xs, out, k)


def s_infogain(Xs, ys, k=K_STAB):
    yv = ys.values
    Hy = _ent(yv)
    ig = []
    for c in Xs.columns:
        xb = _disc(Xs[c].values)
        g = Hy
        for b in np.unique(xb):
            m = xb == b
            g -= m.mean() * _ent(yv[m])
        ig.append(g)
    return _topk(Xs, ig, k)


def s_gainratio(Xs, ys, k=K_STAB):
    yv = ys.values
    Hy = _ent(yv)
    gr = []
    for c in Xs.columns:
        xb = _disc(Xs[c].values)
        g = Hy
        iv = 0.0
        for b in np.unique(xb):
            m = xb == b
            pb = m.mean()
            g -= pb * _ent(yv[m])
            iv -= pb * np.log2(pb + 1e-12)
        gr.append(g / (iv + 1e-12))
    return _topk(Xs, gr, k)


def s_fcbf(Xs, ys, k=K_STAB):
    yv = ys.values
    def su(a, b):
        Ha, Hb = _ent(a), _ent(b)
        Hab = sum(m.mean() * _ent(b[m]) for av in np.unique(a) if (m := (a == av)).any())
        return 2 * (Hb - Hab) / (Ha + Hb + 1e-12)
    return _topk(Xs, [su(_disc(Xs[c].values), yv) for c in Xs.columns], k)


# ---- Embedded Methods (7) ----
def s_lasso(Xs, ys):
    c = LogisticRegressionCV(
        Cs=3, cv=2, penalty="l1", solver="saga", max_iter=200,
        random_state=RANDOM_STATE, n_jobs=1
    ).fit(Xs, ys)
    m = np.abs(c.coef_).sum(0) > 1e-8
    return [Xs.columns[i] for i in range(Xs.shape[1]) if m[i]] or _topk(Xs, np.abs(c.coef_).sum(0))


def s_enet(Xs, ys):
    c = LogisticRegressionCV(
        Cs=3, cv=2, penalty="elasticnet", l1_ratios=[0.5], solver="saga", max_iter=200,
        random_state=RANDOM_STATE, n_jobs=1
    ).fit(Xs, ys)
    m = np.abs(c.coef_).sum(0) > 1e-8
    return [Xs.columns[i] for i in range(Xs.shape[1]) if m[i]] or _topk(Xs, np.abs(c.coef_).sum(0))


def s_rf(Xs, ys):
    sfm = SelectFromModel(
        RandomForestClassifier(n_estimators=30, max_depth=6, random_state=RANDOM_STATE, n_jobs=1),
        threshold="mean",
    ).fit(Xs, ys)
    return [Xs.columns[i] for i in range(Xs.shape[1]) if sfm.get_support()[i]] or _topk(Xs, sfm.estimator_.feature_importances_)


def s_etrees(Xs, ys):
    sfm = SelectFromModel(
        ExtraTreesClassifier(n_estimators=30, max_depth=6, random_state=RANDOM_STATE, n_jobs=1),
        threshold="mean",
    ).fit(Xs, ys)
    return [Xs.columns[i] for i in range(Xs.shape[1]) if sfm.get_support()[i]] or _topk(Xs, sfm.estimator_.feature_importances_)


def s_xgbimp(Xs, ys, k=K_STAB):
    from xgboost import XGBClassifier
    xg = XGBClassifier(
        n_estimators=30, max_depth=4, random_state=RANDOM_STATE, n_jobs=1,
        tree_method="hist", eval_metric="mlogloss"
    ).fit(Xs, ys)
    return _topk(Xs, xg.feature_importances_, k)


def s_lgbm(Xs, ys, k=K_STAB):
    from lightgbm import LGBMClassifier
    m = LGBMClassifier(
        n_estimators=30, max_depth=4, random_state=RANDOM_STATE, n_jobs=1, verbose=-1
    ).fit(Xs, ys)
    return _topk(Xs, m.feature_importances_, k)


def s_dtree(Xs, ys):
    sfm = SelectFromModel(
        DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE), threshold="mean"
    ).fit(Xs, ys)
    return [Xs.columns[i] for i in range(Xs.shape[1]) if sfm.get_support()[i]] or _topk(Xs, sfm.estimator_.feature_importances_)


# ---- Wrappers & Iterative (6) ----
def s_relief(Xs, ys, k=K_STAB):
    from skrebate import ReliefF
    r = ReliefF(
        n_neighbors=min(5, max(1, int(ys.value_counts().min()) - 1)),
        n_jobs=1,
    )
    r.fit(Xs.values, ys.values)
    return _topk(Xs, r.feature_importances_, k)


def s_laplacian(Xs, ys, k=K_STAB):
    from sklearn.neighbors import kneighbors_graph
    Xn = MinMaxScaler().fit_transform(Xs)
    W = kneighbors_graph(
        Xn, n_neighbors=min(5, len(Xn) - 1), mode="distance", n_jobs=1
    ).toarray()
    W = np.where(W > 0, np.exp(-(W**2)), 0.0)
    np.fill_diagonal(W, 0)
    D = W.sum(1)
    sc = []
    for jx in range(Xn.shape[1]):
        f = Xn[:, jx]
        fb = f - (f @ D) / (D.sum() + 1e-12)
        L = (D * fb * fb).sum() - (fb @ (W @ fb))
        sc.append(-(L / ((D * fb * fb).sum() + 1e-12)))
    return _topk(Xs, sc, k)


def s_perm(Xs, ys, k=K_STAB):
    est = RandomForestClassifier(n_estimators=25, max_depth=5, random_state=RANDOM_STATE, n_jobs=1).fit(Xs, ys)
    r = permutation_importance(est, Xs, ys, n_repeats=2, random_state=RANDOM_STATE, n_jobs=1)
    return _topk(Xs, r.importances_mean, k)


def s_rfe(Xs, ys, k=K_STAB):
    r = RFE(
        LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
        n_features_to_select=min(k, Xs.shape[1]), step=0.3
    ).fit(Xs, ys)
    return [Xs.columns[i] for i in range(Xs.shape[1]) if r.support_[i]]


def s_rfecv(Xs, ys):
    r = RFECV(
        LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
        cv=2, scoring="balanced_accuracy", step=0.3, n_jobs=1
    ).fit(Xs, ys)
    return [Xs.columns[i] for i in range(Xs.shape[1]) if r.support_[i]]


def s_sfs(Xs, ys, k=K_STAB):
    sf = _SFS(
        LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
        k_features=min(k, Xs.shape[1]), forward=True, scoring="balanced_accuracy", cv=2, n_jobs=1
    ).fit(Xs.values, ys.values)
    return [Xs.columns[i] for i in sf.k_feature_idx_]


def s_sbs(Xs, ys, k=K_STAB):
    sf = _SFS(
        LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
        k_features=min(k, Xs.shape[1]), forward=False, scoring="balanced_accuracy", cv=2, n_jobs=1
    ).fit(Xs.values, ys.values)
    return [Xs.columns[i] for i in sf.k_feature_idx_]


# ---- Metaheuristics (3) ----
def _meta_stability(Xs, ys, opt_cls):
    from sklearn.metrics import matthews_corrcoef
    from sklearn.model_selection import train_test_split

    Xtr, Xva, ytr, yva = train_test_split(
        Xs, ys, test_size=0.3, stratify=ys, random_state=RANDOM_STATE
    )

    def objective(model, Xtr_f, ytr_f, Xva_f, yva_f):
        model.fit(Xtr_f, ytr_f)
        return -matthews_corrcoef(yva_f, model.predict(Xva_f))

    pop_limit = min(20, Xs.shape[1]) if STABILITY_META_POP is None else int(STABILITY_META_POP)
    pop = max(2, min(pop_limit, Xs.shape[1]))
    opt = opt_cls(
        objective,
        n_iteration=STABILITY_META_ITERS,
        population_size=max(2, int(pop)),
        minimize=True,
    )
    opt.fit(
        LogisticRegression(max_iter=200, class_weight="balanced", random_state=RANDOM_STATE),
        Xtr, ytr, Xva, yva, verbose=False
    )
    return list(opt.best_feature_list)


def s_ga(Xs, ys):
    from zoofs import GeneticOptimization
    return _meta_stability(Xs, ys, GeneticOptimization)


def s_bpso(Xs, ys):
    from zoofs import ParticleSwarmOptimization
    return _meta_stability(Xs, ys, ParticleSwarmOptimization)


def s_bgwo(Xs, ys):
    from zoofs import GreyWolfOptimization
    return _meta_stability(Xs, ys, GreyWolfOptimization)


# ---- Hybrid / Advanced / Ensembles (6) ----
def s_mrmr(Xs, ys, k=K_STAB):
    try:
        from mrmr import mrmr_classif
        return list(mrmr_classif(X=Xs, y=ys, K=min(k, Xs.shape[1]), show_progress=False, n_jobs=1))
    except Exception:
        return _topk(Xs, mutual_info_classif(Xs, ys, random_state=RANDOM_STATE), k)


def s_boruta(Xs, ys):
    from boruta import BorutaPy
    rf = RandomForestClassifier(n_estimators=25, max_depth=4, random_state=RANDOM_STATE, n_jobs=1)
    try:
        b = BorutaPy(rf, n_estimators="auto", random_state=RANDOM_STATE, max_iter=15).fit(Xs.values, ys.values)
        sel = [Xs.columns[i] for i in range(Xs.shape[1]) if b.support_[i]]
        return sel or _topk(Xs, rf.fit(Xs, ys).feature_importances_, K_STAB)
    except Exception:
        return list(Xs.columns[: min(K_STAB, Xs.shape[1])])


def s_shaprank(Xs, ys, k=K_STAB):
    import shap
    from xgboost import XGBClassifier

    xg = XGBClassifier(
        n_estimators=25, max_depth=4, random_state=RANDOM_STATE, n_jobs=1,
        tree_method="hist", eval_metric="logloss"
    ).fit(Xs, ys)

    explanation = shap.TreeExplainer(xg)(Xs)
    sv = np.abs(np.asarray(explanation.values if hasattr(explanation, "values") else explanation))
    nf = Xs.shape[1]

    if sv.ndim == 1:
        imp = sv
    else:
        fa = [ax for ax, sz in enumerate(sv.shape) if sz == nf][-1]
        imp = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != fa)).ravel()

    return _topk(Xs, imp, k)


def s_borutashap(Xs, ys):
    import scipy.stats as _ss
    import shap

    # SciPy and NumPy compatibility shims
    if not hasattr(_ss, "binom_test"):
        _ss.binom_test = (
            lambda x, n=None, p=0.5, alternative="two-sided": _ss.binomtest(
                int(x), int(n), p, alternative=alternative
            ).pvalue
        )
    if not hasattr(np, "NaN"):
        np.NaN = np.nan

    # Patch shap.TreeExplainer.shap_values to convert 3D SHAP outputs into lists for BorutaShap
    if not getattr(shap.TreeExplainer.shap_values, "_is_patched", False):
        orig_shap_values = shap.TreeExplainer.shap_values

        def patched_shap_values(self, X, *args, **kwargs):
            res = orig_shap_values(self, X, *args, **kwargs)
            if isinstance(res, np.ndarray) and res.ndim == 3:
                if res.shape[-1] == 2:
                    return [res[:, :, 0], res[:, :, 1]]
                elif res.shape[0] == 2:
                    return [res[0], res[1]]
                else:
                    return [res[:, :, i] for i in range(res.shape[-1])]
            return res

        patched_shap_values._is_patched = True
        shap.TreeExplainer.shap_values = patched_shap_values

    valid_cols = [c for c in Xs.columns if Xs[c].nunique() > 1]
    if len(valid_cols) == 0:
        return list(Xs.columns[: min(K_STAB, Xs.shape[1])])

    Xs_clean = Xs[valid_cols].copy()

    try:
        from BorutaShap import BorutaShap

        fs = BorutaShap(importance_measure="shap", classification=True)
        fs.fit(
            X=Xs_clean,
            y=ys,
            n_trials=BORUTASHAP_STAB_TRIALS,
            random_state=RANDOM_STATE,
            verbose=False,
        )

        keep = list(getattr(fs, "accepted", []))
        if not keep:
            keep = list(getattr(fs, "tentative", []))

        return keep or list(Xs_clean.columns[: min(K_STAB, Xs_clean.shape[1])])
    except Exception:
        return s_shaprank(Xs, ys, K_STAB)

# Complete Registry (30 Methods)
STAB_ALL = {
    "chi2": s_chi2,
    "anova": s_anova,
    "mutual_info": s_mi,
    "info_gain": s_infogain,
    "gain_ratio": s_gainratio,
    "fisher": s_fisher,
    "variance": s_variance,
    "spearman": s_spearman,
    "fcbf": s_fcbf,
    "reliefF": s_relief,
    "laplacian": s_laplacian,
    "lasso": s_lasso,
    "elasticnet": s_enet,
    "rf_importance": s_rf,
    "extratrees_importance": s_etrees,
    "xgb_importance": s_xgbimp,
    "lgbm_importance": s_lgbm,
    "dtree_importance": s_dtree,
    "permutation": s_perm,
    "rfe": s_rfe,
    "rfecv": s_rfecv,
    "sfs": s_sfs,
    "sbs": s_sbs,
    "ga": s_ga,
    "bpso": s_bpso,
    "bgwo": s_bgwo,
    "mrmr": s_mrmr,
    "boruta": s_boruta,
    "borutashap": s_borutashap,
    "shap_rank": s_shaprank,
}

STAB_METHODS = dict(STAB_ALL)

print(f"Stability panel: {len(STAB_METHODS)} methods active (All 30 enabled and optimized for speed).")


def _stability_data_fingerprint(Xtr, ytr):
    return {
        "n_rows_total": int(len(Xtr)),
        "n_features": int(Xtr.shape[1]),
        "feature_hash": stable_hash(list(Xtr.columns)),
        "target_hash": stable_hash(pd.Series(ytr).astype(str).tolist()),
    }

def _bootstrap_indices(Xtr, ytr, n, k_boots):
    return [
        resample(
            np.arange(len(Xtr)),
            n_samples=n,
            replace=True,
            stratify=ytr,
            random_state=RANDOM_STATE + b,
        )
        for b in range(k_boots)
    ]

def _load_stability_checkpoint(ck, fingerprint, k_boots, methods):
    # Fresh v4.2 checkpoint, or compatible v4.1 checkpoint from the same deterministic design.
    default = {
        "protocol_version": STABILITY_PROTOCOL_VERSION,
        "resumed_from_protocol": None,
        "completed": {},
        "partial": {},
        "failures": {},
        "meta": {},
    }
    if not (RESUME and os.path.exists(ck)):
        return default
    try:
        with open(ck, "r", encoding="utf8") as f:
            loaded = json.load(f)
    except Exception:
        return default

    old_protocol = loaded.get("protocol_version")
    if old_protocol not in {STABILITY_PROTOCOL_VERSION, LEGACY_STABILITY_PROTOCOL_VERSION}:
        return default

    meta = loaded.get("meta", {})
    # v4.1 checkpoints did not contain fingerprints; accept them because bootstrap
    # indices are deterministically regenerated from RANDOM_STATE and row order.
    compatible = (
        not meta
        or (int(meta.get("n_rows_total", fingerprint["n_rows_total"])) == fingerprint["n_rows_total"]
            and int(meta.get("n_features", fingerprint["n_features"])) == fingerprint["n_features"]
            and meta.get("feature_hash", fingerprint["feature_hash"]) == fingerprint["feature_hash"]
            and int(meta.get("k_boots", k_boots)) == k_boots)
    )
    if not compatible:
        print(f"  Stability checkpoint incompatible for {os.path.basename(ck)}; starting a clean v4.2 checkpoint.")
        return default

    loaded.setdefault("completed", {})
    loaded.setdefault("partial", {})
    loaded.setdefault("failures", {})
    loaded["resumed_from_protocol"] = old_protocol if old_protocol != STABILITY_PROTOCOL_VERSION else loaded.get("resumed_from_protocol")
    loaded["protocol_version"] = STABILITY_PROTOCOL_VERSION
    loaded["meta"] = {**meta, **fingerprint, "k_boots": k_boots, "random_state": RANDOM_STATE, "method_count": len(methods)}
    return loaded

def _save_stability_checkpoint(ck, st):
    write_json(ck, st)

def run_stability(ds):
    if ds not in RUNS:
        return pd.DataFrame()

    # Explicitly read the evaluation package only; CV remains at RUNS[ds][10].
    Xtr, Xte, ytr, yte, subsets, provenance, tri, master, state, store = RUNS[ds][:10]
    feat_space = list(Xtr.columns)
    n = min(len(Xtr), int(globals().get("STABILITY_MAX_ROWS", 200)))
    k_boots = int(globals().get("STABILITY_K", 5))
    methods = list(STAB_METHODS.keys())

    if set(methods) != set(EXPECTED_STABILITY_METHODS):
        raise AssertionError(
            f"Stability registry mismatch: expected {EXPECTED_STABILITY_METHOD_COUNT} methods, got {len(methods)}"
        )

    boots_indices = _bootstrap_indices(Xtr, ytr, n, k_boots)
    outdir = os.path.join(BENCH_ROOT, ds, "results")
    os.makedirs(outdir, exist_ok=True)
    ck = os.path.join(outdir, globals().get("STABILITY_CHECKPOINT_NAME", "nb4_stability_v4_checkpoint.json"))
    fingerprint = _stability_data_fingerprint(Xtr, ytr)
    st = _load_stability_checkpoint(ck, fingerprint, k_boots, methods)
    st["meta"] = {**st.get("meta", {}), **fingerprint, "n_rows_bootstrap": n, "k_boots": k_boots,
                   "random_state": RANDOM_STATE, "method_count": len(methods),
                   "method_order": methods}
    _save_stability_checkpoint(ck, st)

    rows = []
    for m in methods:
        completed = st.get("completed", {}).get(m)
        if completed and int(completed.get("n_bootstraps", 0)) == k_boots:
            rows.append(completed)
            continue

        fn = STAB_METHODS[m]
        partial = st.setdefault("partial", {}).setdefault(m, {})
        subs = []

        for bi, idx in enumerate(boots_indices):
            key = str(bi)
            cached = partial.get(key, {})
            if cached.get("status") == "ok" and cached.get("features"):
                subs.append(list(cached["features"]))
                continue

            try:
                Xb = Xtr.iloc[idx].copy()
                yb = ytr.iloc[idx].copy()
                selected = [f for f in list(fn(Xb, yb)) if f in feat_space]
                if not selected:
                    raise ValueError("Selector returned an empty feature subset.")
                partial[key] = {"status": "ok", "features": selected}
                # Remove a stale error for this bootstrap after successful retry.
                st.setdefault("failures", {}).setdefault(m, {}).pop(key, None)
                subs.append(selected)
            except Exception as e:
                partial[key] = {
                    "status": "error",
                    "error_type": type(e).__name__,
                    "error": str(e),
                    "traceback": traceback.format_exc(limit=6),
                }
                st.setdefault("failures", {}).setdefault(m, {})[key] = partial[key]
                print(f"  stability {m} bootstrap {bi+1} ERROR: {type(e).__name__}: {e}")

            _save_stability_checkpoint(ck, st)

        okboots = len(subs)
        rec = {
            "dataset": ds,
            "method": m,
            "jaccard": mean_pairwise_jaccard(subs) if okboots > 1 else np.nan,
            "nogueira": nogueira_stability(subs, feat_space) if okboots > 1 else np.nan,
            "n_bootstraps": okboots,
            "expected_bootstraps": k_boots,
            "stability_rows": n,
            "mean_selected_features": float(np.mean([len(x) for x in subs])) if subs else np.nan,
            "std_selected_features": float(np.std([len(x) for x in subs], ddof=1)) if len(subs) > 1 else 0.0,
            "n_features_reference": len(subsets[m]) if m in subsets else np.nan,
            "status": "PASS" if okboots == k_boots else "INCOMPLETE",
        }

        if rec["status"] == "PASS":
            st["completed"][m] = rec
            st.get("partial", {}).pop(m, None)
            st.get("failures", {}).pop(m, None)
        else:
            st["completed"].pop(m, None)
        rows.append(rec)
        _save_stability_checkpoint(ck, st)

        print(
            f"  {m:<22} Jaccard={rec['jaccard']:.3f}"
            f" Nogueira={rec['nogueira']:.3f} ({okboots}/{k_boots} boots)"
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["method"], keep="last")
        order = {m:i for i,m in enumerate(methods)}
        df["_order"] = df["method"].map(order)
        df = df.sort_values("_order").drop(columns=["_order"]).reset_index(drop=True)
    df.to_csv(os.path.join(outdir, "nb4_stability.csv"), index=False)
    return df


if RUN_STABILITY:
    for ds in list(RUNS):
        try:
            # CRITICAL v4.2 FIX: preserve the existing CV package at index 10.
            cvpack_existing = RUNS[ds][10]
            stab_result = run_stability(ds)
            RUNS[ds] = RUNS[ds][:10] + (cvpack_existing, stab_result)
            print(f"[{ds}] stability stored at RUNS[{ds}][11]; CV preserved at RUNS[{ds}][10].")
        except Exception as e:
            print(f"[{ds}] stability ERROR: {type(e).__name__}: {e}")


## Section 8 — Figures and compactness/performance summary

In [12]:
def make_figures(ds):
    if ds not in RUNS:
        return

    Xtr,Xte,ytr,yte,subsets,provenance,tri,master,state,store = RUNS[ds][:10]
    if len(RUNS[ds]) != EXPECTED_RUNS_LEN:
        print(f'[{ds}] figures skipped: RUNS schema length={len(RUNS[ds])}, expected={EXPECTED_RUNS_LEN}')
        return
    cvpack = RUNS[ds][10]
    stab = RUNS[ds][11]

    outdir = os.path.join(BENCH_ROOT, ds, 'figures')
    os.makedirs(outdir, exist_ok=True)

    best = cvpack[1].copy()
    best = best[best['subset'].isin(CANONICAL_BASELINE_30)]

    if best.empty:
        return

    # Performance ranking
    b = best.sort_values('mcc', ascending=False)

    plt.figure(figsize=(10,7))
    plt.barh(b['subset'][::-1], b['mcc'][::-1])
    plt.xlabel('Held-out test MCC')
    plt.title(f'NB4 — MCC by feature-selection method — {ds}')
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, 'nb4_mcc_ranking.png'),
        dpi=150
    )
    plt.close()

    # Parsimony
    plt.figure(figsize=(8,6))
    plt.scatter(best['n_features'], best['mcc'])

    for _, r in best.iterrows():
        plt.annotate(
            r['subset'],
            (r['n_features'], r['mcc']),
            fontsize=7
        )

    plt.xlabel('Selected features')
    plt.ylabel('Held-out MCC')
    plt.title(f'NB4 — Parsimony vs MCC — {ds}')
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, 'nb4_parsimony.png'),
        dpi=150
    )
    plt.close()

    # Stability plot only if stability results exist
    if not stab.empty:
        s = stab[
            stab['method'].isin(
                CANONICAL_BASELINE_30
            )
        ].dropna(
            subset=['jaccard', 'nogueira']
        )

        if not s.empty:
            plt.figure(figsize=(9,6))
            plt.scatter(
                s['jaccard'],
                s['nogueira']
            )

            for _, r in s.iterrows():
                plt.annotate(
                    r['method'],
                    (r['jaccard'], r['nogueira']),
                    fontsize=7
                )

            plt.xlabel('Mean pairwise Jaccard')
            plt.ylabel('Nogueira stability')
            plt.title(
                f'NB4 — Selection stability — {ds}'
            )
            plt.tight_layout()
            plt.savefig(
                os.path.join(outdir, 'nb4_stability.png'),
                dpi=150
            )
            plt.close()

## Section 9 — Final scientific integrity audit (v4.2)


In [ ]:
AUDITS=[]
for ds in DATASETS:
    if ds not in RUNS:
        AUDITS.append({'dataset':ds,'status':'FAIL','reason':'dataset did not complete'})
        continue

    run = RUNS[ds]
    if len(run) != EXPECTED_RUNS_LEN:
        AUDITS.append({'dataset':ds,'status':'FAIL','reason':f'RUNS schema invalid: expected {EXPECTED_RUNS_LEN}, found {len(run)}'})
        continue

    Xtr,Xte,ytr,yte,subsets,provenance,tri,master,state,store,cvpack,stab = run
    cvdf,best,bestclf=cvpack
    expected_combinations = len(subsets) * EXPECTED_CLASSIFIER_COUNT

    stab_methods = set(stab['method'].astype(str)) if isinstance(stab,pd.DataFrame) and 'method' in stab.columns else set()
    missing_stab = [m for m in EXPECTED_STABILITY_METHODS if m not in stab_methods]
    incomplete_stab = []
    for m in EXPECTED_STABILITY_METHODS:
        if m in stab_methods:
            row = stab.loc[stab['method']==m].iloc[0]
            raw_nboots = pd.to_numeric(row.get('n_bootstraps',0), errors='coerce')
            nboots = int(raw_nboots) if pd.notna(raw_nboots) else 0
            if nboots < STABILITY_K or str(row.get('status','')) != 'PASS':
                incomplete_stab.append(m)

    missing = [m for m in CANONICAL_BASELINE_30 if m not in subsets]
    invalid = {m:[f for f in fs if f not in Xtr.columns] for m,fs in subsets.items() if any(f not in Xtr.columns for f in fs)}
    cv_nan_mcc = int(pd.to_numeric(cvdf.get('cv_mean_mcc',pd.Series(dtype=float)),errors='coerce').isna().sum())
    eval_failures = len(state.get('failures',{}))
    cv_ck = os.path.join(BENCH_ROOT,ds,'results','nb4_cv_checkpoint.json')
    cv_failures = 0
    if os.path.exists(cv_ck):
        try:
            cv_state=json.load(open(cv_ck,encoding='utf8'))
            cv_failures=len(cv_state.get('failures',{}))
        except Exception:
            cv_failures=-1

    core_fail = (missing or invalid or len(master) < expected_combinations or len(cvdf) < expected_combinations
                 or cv_nan_mcc > 0 or eval_failures > 0 or cv_failures > 0)
    stability_complete = (not missing_stab and not incomplete_stab and len(stab_methods)==EXPECTED_STABILITY_METHOD_COUNT)
    status = 'FAIL' if core_fail else ('PASS' if stability_complete else 'PARTIAL')

    rec={
        'dataset':ds,'status':status,'rows':len(Xtr),'test_rows':len(Xte),'features':Xtr.shape[1],
        'expected_baseline_methods':EXPECTED_METHOD_COUNT,'baseline_subsets':sum(m in subsets for m in CANONICAL_BASELINE_30),
        'successful_test_combinations':len(master),'expected_test_combinations':expected_combinations,'evaluation_failures':eval_failures,
        'cv_combinations':len(cvdf),'expected_cv_combinations':expected_combinations,'cv_failures':cv_failures,'cv_nan_mcc':cv_nan_mcc,
        'missing_baselines':missing,'missing_stability':missing_stab,'incomplete_stability':incomplete_stab,
        'stability_methods_found':len(stab_methods),'expected_stability_methods':EXPECTED_STABILITY_METHOD_COUNT,
        'stability_bootstrap_k':STABILITY_K,'invalid_features':invalid,
        'protocol_version':PROTOCOL_VERSION,'runs_schema_version':RUNS_SCHEMA_VERSION,
    }
    write_json(os.path.join(BENCH_ROOT,ds,'results','nb4_master_audit.json'),rec)
    AUDITS.append(rec)

manifest=pd.DataFrame([{k:v for k,v in a.items() if not isinstance(v,(list,dict))} for a in AUDITS])
manifest.to_csv(os.path.join(BENCH_ROOT,'nb4_four_dataset_manifest.csv'),index=False)
print('\n'+'='*78); print('NB4 MASTER SCIENTIFIC INTEGRITY REPORT — v4.2'); print('='*78)
print(manifest.to_string(index=False))
print('\nCanonical baseline methods:',EXPECTED_METHOD_COUNT)
print('Expected stability methods:',EXPECTED_STABILITY_METHOD_COUNT)
print('Datasets:',DATASETS)
print('Primary metric:',PRIMARY_METRIC)
print('CV folds:',CV_FOLDS)
print('Random state:',RANDOM_STATE)
print('Protocol:',PROTOCOL_VERSION)
print('RUNS schema:',RUNS_SCHEMA_VERSION, '(12 items after stability)')
assert EXPECTED_METHOD_COUNT==30
assert EXPECTED_STABILITY_METHOD_COUNT==31
assert len(DATASETS)==4
print('\nMASTER CONFIGURATION GATE: PASS')
print('Unresolved evaluation/CV failures cannot be silently treated as successful results.')
print('Incomplete stability is reported as PARTIAL, not as PASS.')


## Section 10 — Outputs for NB4

Each dataset receives:
- `nb4_master_comparison.csv` — every subset × classifier held-out test result.
- `nb4_classifier_selection_cv.csv` — training-only CV used to select the classifier.
- `nb4_best_per_subset.csv` — one selected classifier per subset, with held-out test metrics.
- `nb4_perfold_cv.csv` — selected-classifier fold-level CV summaries.
- `nb4_stability.csv` — Jaccard + Nogueira stability for the canonical 30 baselines.
- `nb4_friedman_nemenyi.csv` — exploratory within-dataset CV-MCC significance.
- `nb4_master_audit.json` — integrity audit with PASS/PARTIAL/FAIL status.
- checkpoint JSON files for evaluation, CV and stability.

The root receives `nb4_four_dataset_manifest.csv`.

**NB4 uses the same 30-method baseline panel across all four datasets.**

### v4.2 safeguards
- Evaluation and CV checkpoints retain their established v3 protocol identifiers so completed v3 work can be resumed without unnecessary recomputation.
- Stability checkpoints are migrated from compatible v4.1 state to v4.2 metadata automatically.
- `RUNS[ds][10]` is always the CV package and `RUNS[ds][11]` is always the stability DataFrame after Section 7.
- Stability uses deterministic bootstrap indices (`RANDOM_STATE + bootstrap_id`) and checkpoints each bootstrap result.
- A method is promoted to `completed` only after all `STABILITY_K` bootstraps succeed.
- Final audit requires all 30 stability methods to be complete before dataset status becomes PASS.
